In [11]:
import absl.logging
import logging
import requests
import json
import pandas as pd
import os
from datetime import datetime
import io
import time

from download_pdfs import download_pdfs
from grader import grade_paper
from parse import parse_pdfs_in_directory

logging.basicConfig(filename='research_helper.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(filename)s - %(funcName)s - %(message)s')

def search_and_save(query, fields, filename):
    """
    Searches Semantic Scholar API with a given query and saves results to a JSONL file
    in a subdirectory with a timestamp.

    Args:
      query: The search query.
      fields: The fields to retrieve from the API.
      filename: The name of the file to save the results.
    """
    try:
        logging.debug(f"Starting search_and_save for query: {query}")
        # Create subdirectories if they don't exist
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = os.path.join("query_jsons", timestamp)
        os.makedirs(output_dir, exist_ok=True)

        filepath = os.path.join(output_dir, filename)

        url = f"http://api.semanticscholar.org/graph/v1/paper/search/bulk?query={query}&fields={fields}&year=-2025"
        logging.debug(f"Sending initial request to Semantic Scholar API: {url}")

        response = requests.get(url)

        logging.debug(f"Response status code: {response.status_code}")  # Log the status code
        logging.debug(f"Response headers: {response.headers}")

        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        r = response.json()

        logging.debug(f"Response JSON: {r}")

        logging.info(f"Estimated documents for query '{query}': {r.get('total', 'unknown')}")

        retrieved = 0
        with open(filepath, "a") as file:
            while True:
                if "data" in r:
                    retrieved += len(r["data"])
                    logging.debug(f"Retrieved {retrieved} papers for query '{query}'...")
                    for paper in r["data"]:
                        print(json.dumps(paper), file=file)
                if "token" not in r:
                    break

                time.sleep(10)
                
                response = requests.get(f"{url}&token={r['token']}")
                response.raise_for_status()
                r = response.json()

        logging.info(f"Retrieved {retrieved} papers total for query: {query}")

    except requests.exceptions.RequestException as e:
        logging.error(f"Error searching for query '{query}': {e}")
    except Exception as e:
        logging.exception(f"An unexpected error occurred while searching for query '{query}'")


def extrude_external_ids(df):
    """
    Extrudes the content of the 'externalIds' column in a DataFrame.

    Args:
      df: The input DataFrame with an 'externalIds' column containing dictionaries.

    Returns:
      A new DataFrame with the 'externalIds' content extruded into separate columns.
    """
    try:
        external_ids_df = df['externalIds'].apply(pd.Series)
        return pd.concat([df, external_ids_df], axis=1).drop(columns=['externalIds'])
    except Exception as e:
        logging.exception("An unexpected error occurred while extruding external IDs")
        raise  # Re-raise the exception after logging

def extract_urls(df):
    """
    Extracts all URLs from the 'openAccessPdf' column in a DataFrame.

    Args:
      df: The input DataFrame with an 'openAccessPdf' column containing dictionaries.

    Returns:
      A list of URLs extracted from the 'openAccessPdf' column.
    """
    try:
        return df['openAccessPdf'].apply(lambda x: x.get('url')).tolist()
    except Exception as e:
        logging.exception("An unexpected error occurred while extracting URLs")
        raise  # Re-raise the exception after logging

def process_and_download(user_query, queries, output_filename="merged_results.pkl"):
    """
    Processes a list of queries, merges the results, downloads the PDFs, and grades the papers.

    Args:
      user_query: The original user query.
      queries: A list of queries.
      output_filename: The name of the file to save the merged DataFrame.
    """
    try:
        fields = "title,year,abstract,externalIds,url,isOpenAccess,openAccessPdf,fieldsOfStudy"
        dfs = []

        # Create subdirectories if they don't exist
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = os.path.join("query_jsons", timestamp)
        os.makedirs(output_dir, exist_ok=True)

        for query in queries:
            filename = query.lower().replace(" ", "_") + ".jsonl"
            filepath = os.path.join(output_dir, filename)
            search_and_save(query, fields, filepath)
            df = pd.read_json(io.StringIO(filepath), lines=True)
            dfs.append(df)

        merged_df = pd.concat(dfs, axis=0)

        # Grade the papers
        logging.info("Grading papers for relevance...")
        merged_df['relevance_grade'] = merged_df['abstract'].apply(
            lambda abstract: grade_paper(user_query, abstract)
        )
        logging.info("Finished grading papers.")

        # Filter for relevant papers (Apply the mask here)
        merged_df = merged_df[merged_df['relevance_grade'] == True]

        merged_df = extrude_external_ids(merged_df)  # Extrude IDs after filtering
        merged_df.to_pickle(output_filename)
        logging.info(f"Merged DataFrame saved to {output_filename}")

        filtered = merged_df[merged_df["abstract"].notnull()]
        open_access = filtered[filtered["isOpenAccess"] == True]
        final_df = open_access[open_access["openAccessPdf"].notnull()]

        urls = extract_urls(final_df)
        download_pdfs(urls, "pdfs", final_df)

        parse_pdfs_in_directory()

    except Exception as e:
        logging.exception("An unexpected error occurred while processing and downloading")


if __name__ == "__main__":
    absl.logging.set_verbosity(absl.logging.ERROR)
    import sys
    if len(sys.argv) < 2:
        # Log the usage message without the user query
        logging.info("Usage: python search.py <query> [output_filename]")  
        sys.exit(1)

    user_query = sys.argv[1]
    output_filename = sys.argv[2] if len(sys.argv) > 2 else "merged_results.pkl"
    
    # Log that a search is starting
    logging.info("Starting search process...")  

    from queries import query_chain
    results = query_chain.invoke({"query": user_query})
    process_and_download(results.queries, output_filename)

In [12]:
# Call the process_and_download function with a test query
user_query = "neanderthal and carnivores relationship during pleistocene"
queries = ["neanderthal", "carnivores pleistocene"]  # Example sub-queries, or generate using query_chain
process_and_download(user_query, queries, "test_results.pkl")